# Step 3 — Feature engineering

| | |
|---|---|
| **Purpose** | Turn the clean match table into one row per match of pre-kickoff features (`home − away` differences, Elo, market signal, squad value). |
| **Input** | `data/matches_clean.parquet` (from `explore.ipynb`), `data/fpl/<season>/*.csv`. |
| **Output** | `data/model_df.parquet` (git-ignored), `data/team_season_strength.csv` (committed). |
| **Next** | `model.ipynb` |

Loads `data/matches_clean.parquet` (from `explore.ipynb`) and builds per-match features,
every one knowable **before kickoff**. Writes `data/model_df.parquet`.

- **3.1** Reshape to one row per team per match
- **3.2** Rolling form + **EWMA** (continuous decay, no 5-game cliff) — the leak-free core
- **3.3** Venue split, rest days, congestion, head-to-head
- **3.4** Advanced:
  - **a** rolling shot quality, **a-bis** an **expected-goals proxy** (npxG-style, fit by OLS)
  - **b** **margin-aware Elo** with a moving home-field advantage (strength of schedule)
  - **c** opponent-weighted form, derby flag
  - **d** pre-match league position & relegation pressure
  - **e** **market signal** — vig-free (proportional + Shin) 1X2 probabilities, implied
    total goals from the Over/Under line, closing-line movement
- **3.5** Merge team features back as `home_*` / `away_*` / `*_diff`
- **3.6** Leakage checks + `train_ready` view
- **3.7** Squad-strength module from the **FPL Core Insights** player data

Every stage is implemented in **`pl_features.py`** (shared with `predict_upcoming.py`); this
notebook calls it stage by stage and shows the running example, **Man United**, after each.

The rule: a feature for match *k* may only use matches *1…k-1*. Every rolling stat is
`shift(1)` before it is aggregated; Elo is recorded pre-update; odds are the pre-kickoff
price; FPL squad prices are the pre-season snapshot. `home_goals` / `away_goals` are copied
in only as *targets* for the goal model — never read back as inputs.


In [ ]:
import os

import numpy as np
import pandas as pd

import pl_features as F

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

DATA_DIR = "data"
matches = pd.read_parquet(os.path.join(DATA_DIR, "matches_clean.parquet"))
matches = matches.sort_values("Date", kind="mergesort").reset_index(drop=True)
matches["match_id"] = matches.index
print(matches.shape)
matches.head()


## 3.1 Reshape to one row per team per match

Each match -> two rows (home team's view, away team's view). We record what the team
*did* in that match (goals for/against, shots, points). These are match **outcomes** —
3.2 rolls them forward with `shift(1)` so a row never sees its own match.

In [ ]:
tm = F.reshape_team_matches(matches)      # points() gives NaN for a row with no result
print(tm.shape, " (expect", len(matches) * 2, "rows)")
tm.head(4)


## 3.2 Rolling form — the leak-free core

Per team, sorted by date, we want the mean of the team's **previous** N matches:

```
rolling(5).mean() at match k  -> includes match k        (leak)
shift(1) first, then roll     -> mean of matches k-5..k-1  (safe)
```

`_key(...)` collapses multi-column group keys into one Series so
`groupby(key).rolling(...)` returns a clean single-level result to re-align.

Two horizons: **last 5** (recent form) and **season-to-date** (`expanding`, resets each
season). `min_periods=1` -> a value after one prior game; a team's very first ever match
is `NaN`.

In [ ]:
tm = F.add_rolling_form(tm)   # prior_rolling / prior_expanding / prior_ewm over ROLL_STATS, plus games_played
print("window", F.WINDOW, " ewm half-life", F.EWM_HALFLIFE, " stats", F.ROLL_STATS)

# running example: Man United's opening 2015-16 matches — roll5 vs ewm side by side
EXAMPLE_TEAM = "Man United"
(tm[(tm.team == EXAMPLE_TEAM) & (tm.Season == "2015-16")]
   .filter(regex="team|opponent|Date|pts$|pts_roll5|pts_ewm|gd_roll5|gd_ewm|games_played")
   .head(8))


## 3.3 Venue split, rest days, head-to-head

- **venue form** — rolling points, restricted to this venue (home form vs away form)
- **days_rest** — days since this team's previous match; **congestion** — matches in the
  last 14 days
- **head-to-head** — this team's mean points in its last 5 meetings with this specific
  opponent (either venue), keyed on an unordered pair

In [ ]:
tm = F.add_venue_rest_h2h(tm)

(tm[(tm.team == EXAMPLE_TEAM) & (tm.Season == "2015-16")]
   .filter(regex="team|opponent|Date|venue_pts|days_rest|congestion|h2h")
   .head(8))


## 3.4 Advanced features

### 3.4a Rolling shot quality / efficiency

Rolling **shot conversion** (goals / shots on target) and **finishing vs chances**
(goals − a crude 0.3·SoT expected). These regress to the mean, so a team on a hot/cold
streak is flagged. All `shift(1)`-based.

In [ ]:
tm = F.add_shot_quality(tm)

(tm[(tm.team == EXAMPLE_TEAM) & (tm.Season == "2015-16")]
   .filter(regex="team|Date|conv_roll5|sot_rate_roll5|save_pct_roll5")
   .head(8))


### 3.4a-bis  Expected-goals proxy (npxG-style)

Raw shot counts are noisy: a team can take 20 low-quality shots and score once. **Expected
goals** weights each shot by its chance of scoring. We don't have shot coordinates for 25
seasons, but we can build a solid proxy from the two shot columns we *do* have:

$$\widehat{xG} = a + b\cdot(\text{shots} - \text{SoT}) + c\cdot\text{SoT}$$

Coefficients `a, b, c` are fit by ordinary least squares of **actual goals** on
`(off-target shots, on-target shots)` — solved from the 3×3 normal equations, fit **only on
seasons ≤ 2018-19** so nothing downstream of the split leaks in. On-target shots carry
almost all the weight (`c ≈ 0.3`, i.e. ~1 in 3 shots on target is a goal), off-target shots
almost none — exactly the shape a real xG model has.

For **2026-27** the source CSV ships real `HxG / AxG`; we use those directly and fall back
to the proxy everywhere else. Rolling xG-for / xG-against then behave like the other form
features, plus `finishing_roll5 = goals − xG` flags a team riding hot or cold finishing
(it regresses to zero).


In [ ]:
coefs = F.fit_xg_proxy(tm)            # OLS on seasons <= 2018-19 only, solved by Cramer's rule
a, b, c = coefs
print(f"xG proxy fit through {F.XG_FIT_MAX_SEASON}:  xG = {a:.3f} + {b:.4f}*(shots-SoT) + {c:.4f}*SoT")

tm = F.add_xg(tm, matches, coefs)     # real HxG/AxG replace the proxy where the source has them
if "HxG" in matches.columns:
    print(f"real xG available for {int(matches['HxG'].notna().sum())} matches")

(tm[(tm.team == EXAMPLE_TEAM) & (tm.Season == "2015-16")]
   .filter(regex="team|Date|xg_roll5|xga_roll5|xgd_roll5|finishing_roll5")
   .head(8))


### 3.4b Elo rating — margin-aware, with a moving home-field advantage

One number per team that updates after every match and **absorbs strength of schedule**.
Chess Elo, adapted for football with three upgrades over the textbook version:

- **expected score** `E = 1 / (1 + 10**(-(R_home + HFA - R_away)/400))`
- **update** `R += K_eff * (S - E)`, `S ∈ {1, 0.5, 0}` for W / D / L
- **margin-of-victory scaling** (World Football Elo): a 5–0 must move ratings more than a
  1–0. `K_eff = K * sqrt(max(|Δgoals|, 1))`, so a 3-goal win updates ~1.7× a 1-goal win.
- **dynamic home-field advantage**: home edge has drifted down over 25 seasons and
  collapsed in the empty-stadium 2020-21 season. Instead of a fixed `HFA = 60`, we feed
  each season the **home advantage implied by the previous 3 seasons' results**
  (`HFA = -400·log10(1/S̄_home − 1)`, `S̄_home` = league mean home points-rate), so the
  model isn't told to expect a 2005-size home boost in 2021.
- ratings carry across seasons but **regress 25% toward 1500** each season start;
  promoted teams enter at 1500.

Stored per match: both teams' pre-match Elo, their difference, the model's implied
home-win-equivalent `elo_exp_home`, and the `hfa_used` that season.


In [ ]:
print("home-field advantage fed to Elo (rating pts), by season:")
print(pd.Series(F.hfa_by_season(matches)).round(1).to_string())

matches, elo = F.add_elo(matches, return_ratings=True)   # K=20, margin-aware, 25% seasonal regression

_final = pd.Series(elo).sort_values(ascending=False)
print("\ncurrent Elo, top 8:")
print(_final.head(8).round(0).to_string())
print(f"\nMan United: {elo['Man United']:.0f}  (rank {_final.index.get_loc('Man United') + 1})")


### 3.4c Opponent-weighted form + derby flag

- **opponent-weighted form** — like rolling points, but each past result is scaled by the
  opponent's Elo at the time (beating a 1700 side counts more than beating a 1400 side).
  We attach the opponent's pre-match Elo to each `tm` row, then roll `pts * (opp_elo/1500)`.
- **derby flag** — hard-coded rival pairs; derbies suppress home advantage and raise cards.

In [ ]:
tm = F.add_opponent_weighted_form(tm, matches)
matches = F.add_derby(matches)
print("derby matches:", matches.is_derby.sum(),
      "  H/D/A within derbies:",
      matches.loc[matches.is_derby, "FTR"].value_counts().to_dict())


### 3.4e  Market signal — vig-free probabilities & implied goal expectation

The bookmaker's price is the single best pre-match forecast in existence: it aggregates
every model, every injury leak and every stake. Beating *always-home* is a low bar;
**beating the market's log loss is the real test**. We turn the raw odds into features the
tree can use to learn only where public stats spot an inefficiency.

1. **De-vig the 1X2 price.** Raw implied probabilities `1/odds` sum to ~1.05 (the
   "overround"). `mkt_p_*` removes it **proportionally** (divide by the sum) — simple and,
   on this data, within a whisker of optimal. `mkt_pow_*` is a second view that also
   corrects the favourite–longshot bias (`p ∝ (1/odds)^γ`, `γ` fit once on 2000–2018).
2. **Implied total goals.** De-vig the Over/Under 2.5 line, then invert the Poisson tail
   `P(N ≥ 3) = 1 − e^{−λ}(1 + λ + λ²/2)` for the total-goals mean `λ_tot`. Split it with the
   1X2 supremacy → `mkt_xg_home`, `mkt_xg_away` (a clean prior for Dixon-Coles).
3. **Closing-line value.** `clv_* = close_p − open_p` — how far the market moved after the
   open, the sharp-money direction.

All of it is strictly pre-kickoff.


In [ ]:
matches = F.add_market(matches)      # gamma for the power de-vig is fit on seasons <= 2018-19 only

ok = matches[["mkt_H", "mkt_D", "mkt_A"]].notna().all(axis=1)
print(f"market features built — 1X2 de-vig coverage {matches['mkt_p_H'].notna().mean():.1%}, "
      f"O/U coverage {matches['mkt_tot_goals'].notna().mean():.1%}, "
      f"closing coverage {matches['close_p_H'].notna().mean():.1%}, "
      f"power gamma {matches.attrs['power_gamma']:.3f}")
print("mean de-vigged P(H/D/A):",
      matches.loc[ok, ["mkt_p_H", "mkt_p_D", "mkt_p_A"]].mean().round(4).to_dict(),
      " actual:", matches.FTR.value_counts(normalize=True).round(4).to_dict())
matches.loc[matches.Season == "2024-25",
           ["HomeTeam", "AwayTeam", "mkt_p_H", "mkt_p_D", "mkt_p_A",
            "mkt_tot_goals", "mkt_xg_home", "mkt_xg_away"]].head(6)


### 3.4d League position & relegation pressure

Running league table **as it stood before each match** (points, goal difference, rank).
Then two pressure features, more meaningful late in the season:

- `pts_gap_to_safety` — points above/below 18th place
- `pts_gap_to_top4` — points behind 4th

Each requires the table *before* the current match, so we sort by date and compute the
standing incrementally. Round number ~ ceil(games_played_by_league / 10).

In [ ]:
tm = F.add_standings(tm)     # the table as it stood before each match; unplayed rows add nothing

(tm[(tm.team == EXAMPLE_TEAM) & (tm.Season == "2015-16")]
   .filter(regex="team|Season|Date|rank|cum_pts|pts_gap")
   .tail(8))


## 3.5 Merge features back onto the match table

Split `tm` into home rows / away rows, prefix `home_` / `away_`, join on `match_id`.
Then add `*_diff` (home − away) columns.

In [ ]:
model_df = F.merge_features(matches, tm)   # home_* / away_* / *_diff, plus the goal-model targets

print(model_df.shape)
print(f"{sum(c.endswith('_diff') for c in model_df.columns)} *_diff columns, "
      f"{len(F.MARKET_MATCH_FEATS)} market match-level features")
[c for c in model_df.columns if c.endswith("_diff")]


## 3.6 Leakage checks + train-ready view

1. **Independent recompute** of one rolling feature must match exactly.
2. **Elo must be strictly pre-match** — the first match of the dataset has both teams at
   exactly 1500.
3. Drop cold-start rows (either team < 3 games this season) into `train_ready`.
4. Save `data/model_df.parquet`.

In [ ]:
# 1. recompute home_gf_std independently, for a sample of matches
def naive_prior_gf_mean(team, season, upto):
    m = matches[(matches.Season == season) & (matches.Date < upto)
                & ((matches.HomeTeam == team) | (matches.AwayTeam == team))]
    gf = pd.concat([m.loc[m.HomeTeam == team, "FTHG"],
                    m.loc[m.AwayTeam == team, "FTAG"]])
    return gf.mean()

chk = model_df[model_df.home_games_played >= 3].sample(150, random_state=0)
err = max(abs(naive_prior_gf_mean(r.HomeTeam, r.Season, r.Date) - r.home_gf_std)
          for r in chk.itertuples(index=False))
print(f"max |home_gf_std - independent recompute|: {err:.2e}   (want ~0)")

# 1b. same check, spelled out for one Man United home match
mu = model_df[(model_df.HomeTeam == "Man United") & (model_df.Season == "2015-16")
              & (model_df.home_games_played >= 3)].iloc[0]
print(f"\nMan United home vs {mu.AwayTeam} on {mu.Date.date()} "
      f"(game {int(mu.home_games_played) + 1} of the season):")
print(f"  feature home_gf_std      = {mu.home_gf_std:.4f}")
print(f"  independent recompute    = {naive_prior_gf_mean('Man United', '2015-16', mu.Date):.4f}")

# 2. first match: both Elo == 1500
first = model_df.iloc[0]
print(f"\nfirst match Elo: home={first.home_elo_pre:.1f} away={first.away_elo_pre:.1f}  (want 1500 / 1500)")

# 3. train_ready
train_ready = model_df[(model_df.home_games_played >= 3)
                       & (model_df.away_games_played >= 3)].copy()
print(f"\nmodel_df {len(model_df)}  ->  train_ready {len(train_ready)}")
print((train_ready.target.value_counts(normalize=True) * 100).round(1).to_string())
print("\n(model_df.parquet is written at the end of 3.7, after the squad merge)")


## 3.7 Squad-strength module (FPL Core Insights)

Source: **[FPL-Core-Insights](https://github.com/olbauday/FPL-Core-Insights)** — per-season
`players.csv` (squad rosters), `playerstats.csv` (per-gameweek FPL stats), `teams.csv`.
Files in `data/fpl/<season>/`.

FPL player **price** (`now_cost`, in £m) is set before each season from prior-season output
and transfer activity — a clean pre-kickoff proxy for player quality. Per `(Season, team)`:

| feature | how |
|---|---|
| `squad_price_total` | sum of every registered player's pre-season price |
| `squad_price_top11` | price of the 11 most expensive players — the likely first XI |
| `bench_price` | price of squad players ranked 12–20 — depth |
| `gk_price` / `def_price` / `mid_price` / `fwd_price` | best GK, mean of top-5 def, top-5 mid, top-3 fwd — line strength |
| `squad_ppg` | mean prior-season points-per-game across the first XI |

Then merged as `home_* / away_* / *_diff`, plus **`att_edge_diff`** (my forwards vs your
defenders, minus the reverse) — the FM-style line match-up.

### Coverage & leakage

FPL data begins at **2024-25**, so squad features exist for **2024-25, 2025-26, 2026-27**
only; earlier seasons are `NaN` (tree models tolerate it; or train on the covered slice).

Leakage rule: season *S* uses the **GW1 snapshot** of season *S* — prices and
prior-season PPG are fixed before a ball is kicked. We do **not** touch mid-season price
changes, form, or xG. The **2026-27** rosters reflect the completed summer transfer window.


In [ ]:
FPL_DIR = os.path.join(DATA_DIR, "fpl")

fpl_folders = sorted(d for d in os.listdir(FPL_DIR) if os.path.isdir(os.path.join(FPL_DIR, d)))
print(f"{len(fpl_folders)} FPL season folder(s):")
for f in fpl_folders:
    d = F.load_fpl_season(FPL_DIR, f)          # roster + GW1 snapshot, mapped to football-data names
    print(f"  {f}  ->  season {F.fpl_season_label(f)}   ({len(d)} PL players, {d.team.nunique()} clubs)")


In [ ]:
strength = F.build_strength(FPL_DIR, verbose=False)
strength.to_csv(os.path.join(DATA_DIR, "team_season_strength.csv"), index=False)

print(strength.shape, "team-seasons  |  seasons:", sorted(strength.Season.unique()))
print("\n2026-27 (post-transfer-window squads), by first-XI price:")
print(strength[strength.Season == "2026-27"]
      .sort_values("squad_price_top11", ascending=False)
      [["team", "squad_price_top11", "fwd_price", "mid_price", "def_price", "gk_price"]]
      .head(10).to_string(index=False))


In [ ]:
model_df = F.add_squad(model_df, strength)

covered = model_df.home_squad_price_top11.notna().sum()
print(f"squad features merged; {covered}/{len(model_df)} matches covered "
      f"({covered / len(model_df):.0%})")
print("seasons covered:",
      sorted(model_df.loc[model_df.home_squad_price_top11.notna(), "Season"].unique()))

model_df.to_parquet(os.path.join(DATA_DIR, "model_df.parquet"), index=False)
print(f"\nwrote data/model_df.parquet  {model_df.shape}")
